# CCKW 滤波分析 - 比湿数据 (Specific Humidity)

本脚本对比湿数据(hus)进行Kelvin波滤波分析

**处理流程:**
1. 加载3D比湿数据 (time, lev, lat, lon)
2. 对每个垂直层分别应用Kelvin波滤波
3. 保存滤波后的数据到缓存，方便后续直接读取

**日期:** 2026.02.04

In [1]:
# Load packages
import xarray as xr
import numpy as np
import os
import time
import sys
import gc
from pathlib import Path

WAVE_TOOLS_PATH = Path("/work/mh1498/m301257/wave_tools/")
sys.path.insert(0, str(WAVE_TOOLS_PATH.parent))

from wave_tools.filters import CCKWFilter

print("="*70)
print("✅ Packages loaded successfully")
print("="*70)

# Set up directories
CACHE_DIR = "./cache/kelvin_wave_3d/"
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"📁 Cache directory: {CACHE_DIR}")
print("="*70)

✅ Packages loaded successfully
📁 Cache directory: ./cache/kelvin_wave_3d/


In [2]:
# Step 1: 加载比湿数据 (hus)
print("="*70)
print("📊 Loading specific humidity (hus) data")
print("="*70)

# 定义实验和目录映射
exp_dir_map = {
    'CNTL': 'cntl',
    'P4K': 'p4k',
    '4CO2': '4co2'
}

# 数据存储字典
hus_data = {}

for exp, dir_name in exp_dir_map.items():
    print(f"\n{'='*60}")
    print(f"📍 Loading {exp} data")
    print(f"{'='*60}")
    
    # 加载 hus 数据
    hus_file = f"/work/mh1498/m301257/3D_data/{dir_name}/hus_all_levels.nc"
    try:
        hus_ds = xr.open_dataset(hus_file)
        hus_data[exp] = hus_ds['hus'] if 'hus' in hus_ds else hus_ds[list(hus_ds.data_vars)[0]]
        print(f"  ✅ Hus loaded successfully")
        print(f"     Shape: {hus_data[exp].shape}")
        print(f"     Dims: {hus_data[exp].dims}")
        print(f"     Levels: {len(hus_data[exp].lev) if 'lev' in hus_data[exp].dims else 'N/A'}")
    except Exception as e:
        print(f"  ❌ Hus: Failed - {str(e)}")   

print("\n✅ All data loaded successfully")
print("="*70)

📊 Loading specific humidity (hus) data

📍 Loading CNTL data
  ✅ Hus loaded successfully
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: N/A

📍 Loading P4K data
  ✅ Hus loaded successfully
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: N/A

📍 Loading 4CO2 data
  ✅ Hus loaded successfully
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: N/A

✅ All data loaded successfully
  ✅ Hus loaded successfully
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: N/A

📍 Loading P4K data
  ✅ Hus loaded successfully
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: N/A

📍 Loading 4CO2 data
  ✅ Hus loaded successfully
     Shape: (22, 5114, 15, 180)
     Dims: ('level', 'time', 'lat', 'lon')
     Levels: N/A

✅ All data loaded successfully


In [3]:
hus_ds

<xarray.Dataset> Size: 2GB
Dimensions:  (level: 22, time: 5114, lat: 15, lon: 180)
Coordinates:
  * level    (level) int64 176B 31 35 38 41 46 51 55 58 ... 81 83 84 85 87 89 90
  * time     (time) datetime64[ns] 41kB 1980-01-01 1980-01-02 ... 1993-12-31
  * lat      (lat) float64 120B -14.0 -12.0 -10.0 -8.0 ... 8.0 10.0 12.0 14.0
  * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
Data variables:
    hus      (level, time, lat, lon) float64 2GB ...

In [ ]:
# Step 2: 对比湿数据应用Kelvin波滤波（逐层处理）
print("="*70)
print("🌊 Applying Kelvin wave filter to specific humidity data")
print("   Processing each vertical level separately")
print("="*70)

# 滤波参数设置
sel_dict = {
    'time': slice('1980-01-01', '1993-12-31'),
    'lat': slice(-15, 15)
}
wave_name = 'kelvin'
units = 'kg/kg'

# 存储滤波后的数据
kelvin_hus = {}

total_start_time = time.time()

for exp_idx, exp in enumerate(['CNTL', 'P4K', '4CO2'], 1):
    print(f"\n{'='*60}")
    print(f"📍 Processing {exp} ({exp_idx}/3)")
    print(f"{'='*60}")
    
    # 检查缓存
    cache_file = os.path.join(CACHE_DIR, f'kelvin_hus_{exp.lower()}.nc')
    
    if os.path.exists(cache_file):
        print(f"  ♻️  Loading from cache...")
        kelvin_hus[exp] = xr.open_dataarray(cache_file, chunks={'time': 1000})
        print( {kelvin_hus[exp].shape})
        continue
    
    # 如果没有缓存，进行逐层滤波
    exp_start_time = time.time()
    
    try:
        # 获取垂直层数
        hus_exp = hus_data[exp]
        levels = hus_exp.level.values
        n_levels = len(levels)
        
        print(f"  📊 Total levels to process: {n_levels}")
        print(f"  ⏳ Starting level-by-level filtering...")
        
        # 存储每层滤波结果的列表
        filtered_levels = []
        
        # 逐层进行滤波
        for lev_idx, lev in enumerate(levels, 1):
            print(f"\n    🔹 Level {lev_idx}/{n_levels}: {lev:.2f} number level")
            
            # 提取当前层数据
            hus_level = hus_exp.sel(level=lev)
            
            try:
                # 初始化滤波器
                wave_filter = CCKWFilter(
                    ds=hus_level,
                    sel_dict=sel_dict,
                    wave_name=wave_name,
                    units=units,
                    spd=1,
                    n_workers=4,
                    verbose=False  # 减少输出
                )
                
                # 执行滤波步骤
                wave_filter.load_data()
                wave_filter.detrend_data()
                wave_filter.fft_transform()
                wave_filter.apply_filter()
                wave_filter.inverse_fft()
                filtered_level = wave_filter.create_output()
                
                # 添加lev维度
                filtered_level = filtered_level.expand_dims({'level': [lev]})
                filtered_levels.append(filtered_level)
                
                print(f"    ✅ Completed")
                
                # 清理内存
                del wave_filter, filtered_level, hus_level
                gc.collect()
                
            except Exception as e:
                print(f"       ❌ Error at level {lev}: {str(e)}")
                continue
        
        # 合并所有层
        if filtered_levels:
            print(f"\n  🔄 Merging {len(filtered_levels)} filtered levels...")
            filtered_data = xr.concat(filtered_levels, dim='lev')
            
            # 保存到缓存
            print(f"  💾 Saving to cache...")
            encoding = {filtered_data.name or 'hus': {'zlib': True, 'complevel': 4}}
            filtered_data.to_netcdf(cache_file, encoding=encoding)
            
            kelvin_hus[exp] = filtered_data
            
            print(f"     Shape: {filtered_data.shape}")
            print(f"     Memory: {filtered_data.nbytes / 1e9:.2f} GB")
            
            # 清理内存
            del filtered_data, filtered_levels
            gc.collect()
        else:
            print(f"  ❌ No levels were successfully filtered for {exp}")
            
    except Exception as e:
        print(f"  ❌ Error processing {exp}: {str(e)}")
        import traceback
        traceback.print_exc()

total_elapsed = time.time() - total_start_time
print("\n" + "="*70)
print(f"✅ All filtering completed in {total_elapsed/60:.1f} minutes")
print(f"   Processed {len(kelvin_hus)} experiments successfully")
print("="*70)

🌊 Applying Kelvin wave filter to specific humidity data
   Processing each vertical level separately

📍 Processing CNTL (1/3)
  📊 Total levels to process: 22
  ⏳ Starting level-by-level filtering...

    🔹 Level 1/22: 31.00 number level
    ✅ Completed

    🔹 Level 2/22: 35.00 number level
    ✅ Completed

    🔹 Level 2/22: 35.00 number level
    ✅ Completed

    🔹 Level 3/22: 38.00 number level
    ✅ Completed

    🔹 Level 3/22: 38.00 number level
    ✅ Completed

    🔹 Level 4/22: 41.00 number level
    ✅ Completed

    🔹 Level 4/22: 41.00 number level
    ✅ Completed

    🔹 Level 5/22: 46.00 number level
    ✅ Completed

    🔹 Level 5/22: 46.00 number level
    ✅ Completed

    🔹 Level 6/22: 51.00 number level
    ✅ Completed

    🔹 Level 6/22: 51.00 number level
    ✅ Completed

    🔹 Level 7/22: 55.00 number level
    ✅ Completed

    🔹 Level 7/22: 55.00 number level
    ✅ Completed

    🔹 Level 8/22: 58.00 number level
    ✅ Completed

    🔹 Level 8/22: 58.00 number level
    ✅ C

In [ ]:
# Step 3: 验证缓存的滤波数据
print("="*70)
print("🔍 Verifying cached Kelvin-filtered hus data")
print("="*70)

for exp in ['CNTL', 'P4K', '4CO2']:
    cache_file = os.path.join(CACHE_DIR, f'kelvin_hus_{exp.lower()}.nc')
    
    if os.path.exists(cache_file):
        print(f"\n📍 {exp}:")
        try:
            # 加载数据
            data = xr.open_dataarray(cache_file)
            
            # 打印基本信息
            print(f"  ✅ File exists and readable")
            print(f"     Shape: {data.shape}")
            print(f"     Dims: {data.dims}")
            print(f"     Levels: {len(data.level) if 'level' in data.dims else 'N/A'}")
            print(f"     Level range: {float(data.level.min()):.1f} - {float(data.level.max()):.1f} Pa")
            print(f"     File size: {os.path.getsize(cache_file) / 1e9:.2f} GB")
            
            # 检查数据范围
            print(f"     Data range: [{float(data.min()):.6e}, {float(data.max()):.6e}]")
            
        except Exception as e:
            print(f"  ❌ Error reading file: {str(e)}")
    else:
        print(f"\n❌ {exp}: Cache file not found")
        print(f"   Expected: {cache_file}")

